<a href="https://colab.research.google.com/github/akashdas763574/pdf-to-excel-conversion-scrpt/blob/main/pyscript.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pymupdf openpyxl pandas tqdm google-api-python-client google-auth-httplib2 google-auth-oauthlib

In [ ]:
from google.colab import auth

auth.authenticate_user()

print("Google authentication successful")

Google authentication successful


In [ ]:
from googleapiclient.discovery import build
from google.auth import default

creds, _ = default()

drive_service = build(
    "drive",
    "v3",
    credentials=creds
)

FOLDER_ID = "102a6vO6RrGmJY9xqdvr7jU_FrS37rMub"

print("Drive folder connected successfully.")

Drive folder connected successfully.


In [ ]:
results = drive_service.files().list(
    q=f"'{FOLDER_ID}' in parents and trashed=false",
    fields="files(id,name,mimeType)",
    pageSize=1000
).execute()

files = results.get("files", [])

pdf_files = [
    f for f in files
    if f["name"].lower().endswith(".pdf")
]

print("Total PDFs found:", len(pdf_files))

for i, f in enumerate(pdf_files, 1):
    print(i, f["name"])

Total PDFs found: 5
1 80 - Jangalacheruvumalapalli.pdf
2 79 - Kamarajugadda.pdf
3 78 - Chatragadda.pdf
4 77 - CHATRAGADDA.pdf
5 76 - Penumudi.pdf


In [ ]:
from pathlib import Path
from googleapiclient.http import MediaIoBaseDownload

PDF_FOLDER = Path("/content/SSR_PDF")
PDF_FOLDER.mkdir(exist_ok=True)

for i, pdf in enumerate(pdf_files, 1):

    output_path = PDF_FOLDER / pdf["name"]

    request = drive_service.files().get_media(
        fileId=pdf["id"]
    )

    with open(output_path, "wb") as fh:

        downloader = MediaIoBaseDownload(
            fh,
            request
        )

        done = False

        while not done:
            status, done = downloader.next_chunk()

    print(
        f"[{i}/{len(pdf_files)}] {pdf['name']}"
    )

print("\nAll PDFs downloaded.")

[1/5] 80 - Jangalacheruvumalapalli.pdf
[2/5] 79 - Kamarajugadda.pdf
[3/5] 78 - Chatragadda.pdf
[4/5] 77 - CHATRAGADDA.pdf
[5/5] 76 - Penumudi.pdf

All PDFs downloaded.


In [ ]:
import fitz
from pathlib import Path

pdf_list = sorted(
    PDF_FOLDER.glob("*.pdf")
)

print("Testing PDF text layer...\n")

for pdf_path in pdf_list:

    doc = fitz.open(pdf_path)

    total_chars = 0
    pages_with_text = 0

    for page in doc:

        text = page.get_text("text")

        if text.strip():
            pages_with_text += 1
            total_chars += len(text)

    print(
        f"{pdf_path.name}"
    )

    print(
        f"Pages: {len(doc)} | "
        f"Text pages: {pages_with_text} | "
        f"Characters: {total_chars}"
    )

    doc.close()

Testing PDF text layer...

76 - Penumudi.pdf
Pages: 32 | Text pages: 0 | Characters: 0
77 - CHATRAGADDA.pdf
Pages: 30 | Text pages: 0 | Characters: 0
78 - Chatragadda.pdf
Pages: 30 | Text pages: 0 | Characters: 0
79 - Kamarajugadda.pdf
Pages: 32 | Text pages: 0 | Characters: 0
80 - Jangalacheruvumalapalli.pdf
Pages: 15 | Text pages: 0 | Characters: 0


In [ ]:
import fitz

test_pdf = pdf_list[0]

doc = fitz.open(test_pdf)

for page_number in range(
    min(3, len(doc))
):

    page = doc[page_number]

    text = page.get_text(
        "text"
    )

    print("=" * 80)
    print(
        "PAGE:",
        page_number + 1
    )
    print("=" * 80)

    print(
        text[:5000]
    )

doc.close()

PAGE: 1

PAGE: 2

PAGE: 3



In [ ]:
import fitz
import re
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm


COLUMNS = [
    "Age",
    "Serial Number",
    "Candidate Name",
    "ID Number/EPIC",
    "Relative Type",
    "Relative Name",
    "House Number",
    "Gender",
    "Source Page"
]


def clean(value):

    if value is None:
        return ""

    value = str(value)

    value = value.replace(
        "\u00a0",
        " "
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


def upper(value):

    return clean(value).upper()


def get_serial(text):

    patterns = [
        r"(?:Serial\s*No\.?|Serial\s*Number|S\.?\s*No\.?)\s*[:\-]?\s*(\d+)",
        r"(?:Sl\.?\s*No\.?)\s*[:\-]?\s*(\d+)"
    ]

    for p in patterns:

        m = re.search(
            p,
            text,
            re.I
        )

        if m:
            return int(m.group(1))

    return None


def get_age(text):

    patterns = [
        r"Age\s*[:\-]?\s*(\d{1,3})",
        r"\bAge\s+(\d{1,3})\b"
    ]

    for p in patterns:

        m = re.search(
            p,
            text,
            re.I
        )

        if m:

            age = int(
                m.group(1)
            )

            if 18 <= age <= 120:
                return age

    return None


def get_gender(text):

    u = text.upper()

    if re.search(
        r"\bFEMALE\b",
        u
    ):
        return "Female"

    if re.search(
        r"\bTHIRD\s+GENDER\b",
        u
    ):
        return "Third Gender"

    if re.search(
        r"\bMALE\b",
        u
    ):
        return "Male"

    return ""


def get_epic(text):

    patterns = [
        r"\b[A-Z]{3}\d{7}\b",
        r"\b[A-Z]{2,4}\d{6,8}\b"
    ]

    u = text.upper()

    for p in patterns:

        matches = re.findall(
            p,
            u
        )

        if matches:
            return matches[0]

    return ""


def get_value(
    text,
    labels
):

    lines = text.splitlines()

    for i, line in enumerate(lines):

        current = clean(line)

        for label in labels:

            if re.search(
                re.escape(label),
                current,
                re.I
            ):

                value = re.sub(
                    re.escape(label),
                    "",
                    current,
                    flags=re.I
                )

                value = value.strip(
                    " :-.\t"
                )

                if value:
                    return value

                # Wrapped value
                if i + 1 < len(lines):

                    nxt = clean(
                        lines[i + 1]
                    )

                    if nxt:
                        return nxt

    return ""


def get_name(text):

    return upper(
        get_value(
            text,
            [
                "Name:",
                "Name :"
            ]
        )
    )


def get_relative_type(text):

    u = text.upper()

    # Order matters
    if re.search(
        r"\bHUSBAND\b",
        u
    ):
        return "Husband"

    if re.search(
        r"\bFATHER\b",
        u
    ):
        return "Father"

    if re.search(
        r"\bMOTHER\b",
        u
    ):
        return "Mother"

    if re.search(
        r"\bWIFE\b",
        u
    ):
        return "Wife"

    return ""


def get_relative_name(text):

    return upper(
        get_value(
            text,
            [
                "Father Name:",
                "Father Name :",
                "Mother Name:",
                "Mother Name :",
                "Husband Name:",
                "Husband Name :",
                "Wife Name:",
                "Wife Name :",
                "Relative Name:",
                "Relative Name :"
            ]
        )
    )


def get_house(text):

    return clean(
        get_value(
            text,
            [
                "House Number:",
                "House Number :",
                "House No:",
                "House No :",
                "House No."
            ]
        )
    )


def extract_pdf_fast(pdf_path):

    doc = fitz.open(
        pdf_path
    )

    records = []

    for page_index in range(
        len(doc)
    ):

        page = doc[
            page_index
        ]

        page_number = (
            page_index + 1
        )

        text = page.get_text(
            "text"
        )

        if not text.strip():
            continue

        # Split around likely serial-number starts.
        blocks = re.split(
            r"(?=(?:Serial\s*(?:No\.?|Number)|S\.?\s*No\.?|Sl\.?\s*No\.?)\s*[:\-]?\s*\d+)",
            text,
            flags=re.I
        )

        for block in blocks:

            block = clean(block)

            if not block:
                continue

            serial = get_serial(
                block
            )

            age = get_age(
                block
            )

            # A voter block normally has
            # serial and age.
            if serial is None:
                continue

            if age is None:
                continue

            record = {
                "Age": age,
                "Serial Number": serial,
                "Candidate Name": get_name(block),
                "ID Number/EPIC": get_epic(block),
                "Relative Type": get_relative_type(block),
                "Relative Name": get_relative_name(block),
                "House Number": get_house(block),
                "Gender": get_gender(block),
                "Source Page": page_number
            }

            records.append(
                record
            )

    doc.close()

    df = pd.DataFrame(
        records,
        columns=COLUMNS
    )

    if df.empty:
        return df

    df["Candidate Name"] = (
        df["Candidate Name"]
        .fillna("")
        .astype(str)
        .str.upper()
    )

    df["Relative Name"] = (
        df["Relative Name"]
        .fillna("")
        .astype(str)
        .str.upper()
    )

    df["Age"] = pd.to_numeric(
        df["Age"],
        errors="coerce"
    )

    df["Serial Number"] = pd.to_numeric(
        df["Serial Number"],
        errors="coerce"
    )

    df["House Number"] = (
        df["House Number"]
        .fillna("")
        .astype(str)
    )

    df = df.sort_values(
        [
            "Age",
            "Serial Number"
        ],
        ascending=True,
        kind="stable"
    )

    return df.reset_index(
        drop=True
    )

In [ ]:
import time

test_pdf = pdf_list[0]

start = time.time()

df_test = extract_pdf_fast(
    test_pdf
)

elapsed = time.time() - start

print()
print("=" * 70)
print("FAST EXTRACTION TEST")
print("=" * 70)

print(
    "PDF:",
    test_pdf.name
)

print(
    "Records extracted:",
    len(df_test)
)

print(
    "Time:",
    round(elapsed, 2),
    "seconds"
)

print()
print(df_test.head(10).to_string(
    index=False
))

In [ ]:
import time

FAST_OUTPUT = Path(
    "/content/SSR_EXCEL_FAST"
)

FAST_OUTPUT.mkdir(
    exist_ok=True
)

all_results = {}

start_all = time.time()

for i, pdf_path in enumerate(
    pdf_list,
    1
):

    start = time.time()

    print()
    print(
        "=" * 70
    )

    print(
        f"[{i}/{len(pdf_list)}]",
        pdf_path.name
    )

    df = extract_pdf_fast(
        pdf_path
    )

    all_results[
        pdf_path.name
    ] = df

    elapsed = (
        time.time() - start
    )

    print(
        "Records:",
        len(df)
    )

    print(
        "Time:",
        round(elapsed, 2),
        "seconds"
    )

total_time = (
    time.time() - start_all
)

print()
print(
    "=" * 70
)

print(
    "TOTAL PROCESSING TIME:",
    round(total_time, 2),
    "seconds"
)


[1/5] 76 - Penumudi.pdf
Records: 0
Time: 0.02 seconds

[2/5] 77 - CHATRAGADDA.pdf
Records: 0
Time: 0.02 seconds

[3/5] 78 - Chatragadda.pdf
Records: 0
Time: 0.02 seconds

[4/5] 79 - Kamarajugadda.pdf
Records: 0
Time: 0.02 seconds

[5/5] 80 - Jangalacheruvumalapalli.pdf
Records: 0
Time: 0.01 seconds

TOTAL PROCESSING TIME: 0.08 seconds
